In [53]:
import sys

# instalar en el entorno activo
!{sys.executable} -m pip install dash plotly wordcloud pywaffle altair seaborn matplotlib

# imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import plotly.express as px
import altair as alt
import ast

from dash import Dash, dcc, html
from wordcloud import WordCloud
from pywaffle import Waffle

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [54]:
# 1. Cargar datos
movies = pd.read_csv("Datos/movies.csv")
ratings = pd.read_csv("Datos/ratings.csv")
links = pd.read_csv("Datos/links.csv")

metadata = pd.read_csv("archive/movies_metadata.csv", low_memory=False)

# 2. Preparar IDs (IMDb)
metadata["imdb_id_clean"] = metadata["imdb_id"].str.replace("tt", "", regex=False)
metadata["imdb_id_clean"] = pd.to_numeric(metadata["imdb_id_clean"], errors="coerce")

links["imdbId"] = pd.to_numeric(links["imdbId"], errors="coerce")

# 3. Eliminar NaN en claves
metadata = metadata.dropna(subset=["imdb_id_clean"])
links = links.dropna(subset=["imdbId"])

# 4. Corrección de tipos
metadata["imdb_id_clean"] = metadata["imdb_id_clean"].astype(int)
links["imdbId"] = links["imdbId"].astype(int)

# 5. Merge Kaggle + MovieLens (metadata)
df_kaggle = metadata.merge(
    links,
    left_on="imdb_id_clean",
    right_on="imdbId"
)

# 6. Agregar ratings
ratings_agg = (
    ratings.groupby("movieId")
    .agg(
        rating_mean=("rating", "mean"),
        rating_count=("rating", "count")
    )
    .reset_index()
)

# 7. Merge final
df_final = (
    df_kaggle
    .merge(movies, on="movieId")
    .merge(ratings_agg, on="movieId")
)

# 8. Verificación
print("Shape final:", df_final.shape)
print("Películas únicas:", df_final["movieId"].nunique())

df_final.head()

Shape final: (9510, 32)
Películas únicas: 9507


,adult,belongs_to_collection,budget,genres_x,homepage,id,imdb_id,original_language,original_title,overview,...,vote_average,vote_count,imdb_id_clean,movieId,imdbId,tmdbId,title_y,genres_y,rating_mean,rating_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,7.7,5415.0,114709,1,114709,862.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.920930,215
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,6.9,2413.0,113497,2,113497,8844.0,Jumanji (1995),Adventure|Children|Fantasy,3.431818,110
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,6.5,92.0,113228,3,113228,15602.0,Grumpier Old Men (1995),Comedy|Romance,3.259615,52
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,6.1,34.0,114885,4,114885,31357.0,Waiting to Exhale (1995),Comedy|Drama|Romance,2.357143,7
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,5.7,173.0,113041,5,113041,11862.0,Father of the Bride Part II (1995),Comedy,3.071429,49


In [55]:
# Eliminar columnas innecesarias

columns_to_drop = [
    "belongs_to_collection",
    "id",
    "homepage",
    "imdb_id",
    "status",
    "video",
    "tmdbId",
    "rating_mean",
    "rating_count"   
]

for col in columns_to_drop:
    if col in df_final.columns:
        df_final = df_final.drop(columns=[col])

# Verificación
print("Columnas restantes:", df_final.columns.tolist())

Columnas restantes: ['adult', 'budget', 'genres_x', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'tagline', 'title_x', 'vote_average', 'vote_count', 'imdb_id_clean', 'movieId', 'imdbId', 'title_y', 'genres_y']


In [56]:
# Corregir production_companies

df_final["production_companies"] = df_final["production_companies"].str.findall(r"'([^']*)'").str[1]

# verificación
df_final["production_companies"].head()

0                   Pixar Animation Studios
1                          TriStar Pictures
2                              Warner Bros.
3    Twentieth Century Fox Film Corporation
4                     Sandollar Productions
Name: production_companies, dtype: object

In [57]:
# production_countries → nuevas columnas
if "production_countries" in df_final.columns:
    
    extracted_countries = df_final["production_countries"].str.findall(r"'([^']*)'")
    
    # Código ISO del país
    df_final["countries_iso"] = extracted_countries.str[1]
    
    # Nombre completo del país
    df_final["countries_name"] = extracted_countries.str[3]

# spoken_languages → nombre del idioma
if "spoken_languages" in df_final.columns:
    
    extracted_languages = df_final["spoken_languages"].str.findall(r"'([^']*)'")
    
    # Nombre del idioma
    df_final["spoken_languages"] = extracted_languages.str[3]

# Verificación
df_final[[
    "production_countries",
    "countries_iso",
    "countries_name",
    "spoken_languages"
]].head()

,production_countries,countries_iso,countries_name,spoken_languages
0,"[{'iso_3166_1': 'US', 'name': 'United States o...",US,United States of America,English
1,"[{'iso_3166_1': 'US', 'name': 'United States o...",US,United States of America,English
2,"[{'iso_3166_1': 'US', 'name': 'United States o...",US,United States of America,English
3,"[{'iso_3166_1': 'US', 'name': 'United States o...",US,United States of America,English
4,"[{'iso_3166_1': 'US', 'name': 'United States o...",US,United States of America,English


In [58]:
# =========================
# Corregir géneros y eliminar columnas innecesarias
# =========================

# separar géneros múltiples y expandir filas
if "genres_y" in df_final.columns:
    df_final["genres_y"] = df_final["genres_y"].str.split("|")
    df_final = df_final.explode("genres_y")
    df_final = df_final.rename(columns={"genres_y": "genre"})

# eliminar valores no útiles
if "genre" in df_final.columns:
    df_final = df_final[df_final["genre"].notna()]
    df_final = df_final[df_final["genre"] != "(no genres listed)"]

# eliminar columnas innecesarias
columns_to_drop = ["genres_x", "belongs_to_collection"]

for col in columns_to_drop:
    if col in df_final.columns:
        df_final = df_final.drop(columns=[col])

# verificación
print("Número de géneros:", df_final["genre"].nunique())
print(sorted(df_final["genre"].dropna().unique()))
df_final[["movieId", "genre"]].head(10)

Número de géneros: 19
['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


,movieId,genre
0,1,Adventure
0,1,Animation
0,1,Children
0,1,Comedy
0,1,Fantasy
1,2,Adventure
1,2,Children
1,2,Fantasy
2,3,Comedy
2,3,Romance


In [59]:
# Eliminar columnas innecesarias 2

columns_to_drop = [
    "production_countries",
    "title_y"   
]

for col in columns_to_drop:
    if col in df_final.columns:
        df_final = df_final.drop(columns=[col])

# Verificación
print("Columnas restantes:", df_final.columns.tolist())

Columnas restantes: ['adult', 'budget', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'tagline', 'title_x', 'vote_average', 'vote_count', 'imdb_id_clean', 'movieId', 'imdbId', 'genre', 'countries_iso', 'countries_name']


In [60]:
# =========================
# 1. Convertir release_date a fecha
# =========================
if "release_date" in df_final.columns:
    df_final["release_date"] = pd.to_datetime(df_final["release_date"], errors="coerce")

# =========================
# 2. Crear release_year desde release_date
# =========================
if "release_date" in df_final.columns:
    df_final["release_year"] = df_final["release_date"].dt.year

# =========================
# 3. Convertir release_year a int
# =========================
if "release_year" in df_final.columns:
    df_final = df_final.dropna(subset=["release_year"])
    df_final["release_year"] = df_final["release_year"].astype(int)

# =========================
# 4. Convertir columnas numéricas importantes
# =========================
for col in ["budget", "popularity"]:
    if col in df_final.columns:
        df_final[col] = pd.to_numeric(df_final[col], errors="coerce")

# =========================
# 5. Verificación
# =========================
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21563 entries, 0 to 9509
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   adult                 21563 non-null  object        
 1   budget                21563 non-null  int64         
 2   original_language     21563 non-null  object        
 3   original_title        21563 non-null  object        
 4   overview              21534 non-null  object        
 5   popularity            21563 non-null  float64       
 6   poster_path           21560 non-null  object        
 7   production_companies  20379 non-null  object        
 8   release_date          21563 non-null  datetime64[ns]
 9   revenue               21563 non-null  float64       
 10  runtime               21563 non-null  float64       
 11  spoken_languages      21380 non-null  object        
 12  tagline               17030 non-null  object        
 13  title_x               

In [61]:
# =========================
# 1. Convertir columnas object a string
# =========================
for col in df_final.columns:
    if df_final[col].dtype == "object":
        df_final[col] = df_final[col].astype("string")

# =========================
# 2. Crear URL de posters
# =========================
if "poster_path" in df_final.columns:
    base_url = "https://image.tmdb.org/t/p/w500"
    
    df_final["poster_url"] = base_url + df_final["poster_path"]

# =========================
# 3. Verificación
# =========================
df_final[["poster_path", "poster_url"]].head()

,poster_path,poster_url
0,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,https://image.tmdb.org/t/p/w500/rhIRbceoE9lR4v...
0,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,https://image.tmdb.org/t/p/w500/rhIRbceoE9lR4v...
0,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,https://image.tmdb.org/t/p/w500/rhIRbceoE9lR4v...
0,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,https://image.tmdb.org/t/p/w500/rhIRbceoE9lR4v...
0,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,https://image.tmdb.org/t/p/w500/rhIRbceoE9lR4v...


In [62]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21563 entries, 0 to 9509
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   adult                 21563 non-null  string        
 1   budget                21563 non-null  int64         
 2   original_language     21563 non-null  string        
 3   original_title        21563 non-null  string        
 4   overview              21534 non-null  string        
 5   popularity            21563 non-null  float64       
 6   poster_path           21560 non-null  string        
 7   production_companies  20379 non-null  string        
 8   release_date          21563 non-null  datetime64[ns]
 9   revenue               21563 non-null  float64       
 10  runtime               21563 non-null  float64       
 11  spoken_languages      21380 non-null  string        
 12  tagline               17030 non-null  string        
 13  title_x               

In [63]:
df_final.to_csv("movies_final.csv", index=False)